# Late Interaction using ColBert
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring serverside.<br>
However, turbopuffer doesn't support multi-vector documents today.

Since turbopuffer does not natively support multi-vector documents, a practical ColBERT implementation today is a two-stage approximation:
* dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation desciions:
* At index time, ColBERT is run on Every document and token vectors vectors are stored as JSON attribute on each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer trturns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.
Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors trabel with the document, no lookups or joins.
* Storing JSOB blobs is cheaper in turbopuffer's storage model.


In [ ]:
import turbopuffer
from dotenv import load_dotenv
import os

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-central1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-turbopuffer')

### Dateset: Load Quora Question Paris from HuggingFace

In [12]:
import pandas as pd

df_qpair = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair/train-00000-of-00001.parquet")
df_qpair.count()

anchor      149263
positive    149263
dtype: int64

In [13]:
df_qpair.head(100)

,anchor,positive
0,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan..."
1,How can I be a good geologist?,What should I do to be a great geologist?
2,How do I read and find my YouTube comments?,How can I see all my Youtube comments?
3,What can make Physics easy to learn?,How can you make physics easy to learn?
4,What was your first sexual experience like?,What was your first sexual experience?
...,...,...
95,Will Modi win in 2019?,Can Narendra Modi become Prime Minister of Ind...
96,"What exactly is the ""Common Core Initiative/St...",What are the pros and cons of the Common Core ...
97,How do I choose a journal to publish my paper?,Where do I publish my paper?
98,What are your New Year's resolutions for 2017?,What is your creative New Year's resolution fo...


The dataset has ~150K questions pairs `anchor` and `positive`. We will:
* take a subset of 1000 questions pairs
* use `positive` to build corpus, create dense embeddings + `token_vector` attribute in turbopuffer
* use a few `anchor` questions to compare the search performance of **dense retreival** vs **late interaction**

Succes metric would be "did the matching positive for given anchor rank #1" using each technique.

In [24]:
df_corpus = df_qpair['positive'][:1000]
print("Corpus length is", df_corpus.count(), "\nCorpus Sample:")
df_corpus.head()

Corpus length is 1000 
Corpus Sample:


0    I'm a triple Capricorn (Sun, Moon and ascendan...
1            What should I do to be a great geologist?
2               How can I see all my Youtube comments?
3              How can you make physics easy to learn?
4               What was your first sexual experience?
Name: positive, dtype: str